# Feature Encoding
* Label Encoding
* One-Hot
* Ordinal
* Frequency
* Target
* Binary encoding

In [ ]:
import pandas as pd
data = {
    "City": ["Kochi", "Kottayam", "Kochi", "Trivandrum", "Kollam"],
    "Education": ["UG", "PG", "UG", "PhD", "UG"],
    "Gender": ["M", "F", "M", "F", "M"],
    "Salary": [30000, 45000, 32000, 60000, 28000]
}

df = pd.DataFrame(data)
df.head()

,City,Education,Gender,Salary
0,Kochi,UG,M,30000
1,Kottayam,PG,F,45000
2,Kochi,UG,M,32000
3,Trivandrum,PhD,F,60000
4,Kollam,UG,M,28000


## 1) Label Encoding
**What it does:** Converts each category to an integer label (0,1,2...).

**When to use:** Good for binary categories or tree-based models (Decision Trees, Random Forests, XGBoost). Avoid for nominal categories when using linear models because integer values imply order.

**Tip:** Keep `LabelEncoder` instances for each column if you need to inverse-transform later.

In [ ]:
from sklearn.preprocessing import LabelEncoder  # algorithms cannot handle the categorical variables unless they are converted into a numerical value.

le_city = LabelEncoder()
le_gender = LabelEncoder()

df['City_LE'] = le_city.fit_transform(df['City'])
df['Gender_LE'] = le_gender.fit_transform(df['Gender'])
df.head()

,City,Education,Gender,Salary,City_LE,Gender_LE
0,Kochi,UG,M,30000,0,1
1,Kottayam,PG,F,45000,2,0
2,Kochi,UG,M,32000,0,1
3,Trivandrum,PhD,F,60000,3,0
4,Kollam,UG,M,28000,1,1


## 2) One-Hot Encoding (Pandas)
**What it does:** Creates a binary column for each category (0/1). Safe for linear models because it does not impose ordering.

**When to use:** Nominal categorical variables with relatively few unique values.

**Tip:** Use `drop_first=True` to avoid the dummy variable trap in linear models (reduces multicollinearity). For high-cardinality features, OHE may explode number of columns.

In [ ]:
df_ohe = pd.get_dummies(df, columns=['City'], drop_first=True)
df_ohe.head()

,Education,Gender,Salary,City_LE,Gender_LE,City_Kollam,City_Kottayam,City_Trivandrum
0,UG,M,30000,0,1,False,False,False
1,PG,F,45000,2,0,False,True,False
2,UG,M,32000,0,1,False,False,False
3,PhD,F,60000,3,0,False,False,True
4,UG,M,28000,1,1,True,False,False


## 2.1) One-Hot Encoding (Scikit-Learn)
**What it does:** Same as Pandas OHE but returns an array. Useful inside pipelines because it integrates with `ColumnTransformer`.

**Tip:** Use `handle_unknown='ignore'` in production pipelines to avoid errors when unseen categories appear in test data.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = ohe.fit_transform(df[['City']])

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(['City']))
pd.concat([df, encoded_df], axis=1).head()

,City,Education,Gender,Salary,City_LE,Gender_LE,City_Kochi,City_Kollam,City_Kottayam,City_Trivandrum
0,Kochi,UG,M,30000,0,1,1.0,0.0,0.0,0.0
1,Kottayam,PG,F,45000,2,0,0.0,0.0,1.0,0.0
2,Kochi,UG,M,32000,0,1,1.0,0.0,0.0,0.0
3,Trivandrum,PhD,F,60000,3,0,0.0,0.0,0.0,1.0
4,Kollam,UG,M,28000,1,1,0.0,1.0,0.0,0.0


## 3) Ordinal Encoding
**What it does:** Maps ordered categories to integers following a user-specified order.

**When to use:** Education level (UG < PG < PhD) or rating scales (Low < Medium < High).

**Tip:** Always provide the `categories` order explicitly to avoid incorrect default ordering.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

order = [['UG', 'PG', 'PhD']]
oe = OrdinalEncoder(categories=order)
df['Education_Ordinal'] = oe.fit_transform(df[['Education']])
df.head()

,City,Education,Gender,Salary,City_LE,Gender_LE,Education_Ordinal
0,Kochi,UG,M,30000,0,1,0.0
1,Kottayam,PG,F,45000,2,0,1.0
2,Kochi,UG,M,32000,0,1,0.0
3,Trivandrum,PhD,F,60000,3,0,2.0
4,Kollam,UG,M,28000,1,1,0.0


### 4) Frequency Encoding
**What it does:** Replaces categories with their frequency (count) in the dataset.

**When to use:** Useful for high-cardinality features where OHE would produce too many columns.

**Tip:** Frequency encoding preserves information about how common a category is but loses the actual label semantics.

In [ ]:
freq_map = df['City'].value_counts().to_dict()
df['City_Freq'] = df['City'].map(freq_map)
df.head()

,City,Education,Gender,Salary,City_LE,Gender_LE,Education_Ordinal,City_Freq
0,Kochi,UG,M,30000,0,1,0.0,2
1,Kottayam,PG,F,45000,2,0,1.0,1
2,Kochi,UG,M,32000,0,1,0.0,2
3,Trivandrum,PhD,F,60000,3,0,2.0,1
4,Kollam,UG,M,28000,1,1,0.0,1


## 5) Target Encoding
**What it does:** Replaces categories with the mean of the target variable for that category (e.g., mean Salary per City).

**When to use:** High-cardinality categorical features where category-target relationship is informative.

**Warning:** Risk of target leakage and overfitting. Use K-fold target encoding, smoothing, or fit only on training data and apply to validation/test data.

**Tip:** The `category_encoders.TargetEncoder` has options for smoothing — use in cross-validated pipelines for safety.

In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.0 MB/s eta 0:00:00


In [ ]:
import category_encoders as ce

target_enc = ce.TargetEncoder(cols=['City'])
df['City_Target'] = target_enc.fit_transform(df['City'], df['Salary'])
df.head()

,City,Education,Gender,Salary,City_LE,Gender_LE,Education_Ordinal,City_Freq,City_Target
0,Kochi,UG,M,30000,0,1,0.0,2,37865.191481
1,Kottayam,PG,F,45000,2,0,1.0,1,39780.650846
2,Kochi,UG,M,32000,0,1,0.0,2,37865.191481
3,Trivandrum,PhD,F,60000,3,0,2.0,1,41732.277962
4,Kollam,UG,M,28000,1,1,0.0,1,37568.806782


## 6) Binary Encoding
**What it does:** Converts categories into binary digits based on integer label, then creates separate columns for each binary digit. It's a compromise between Label and One-Hot encoding.

**When to use:** High-cardinality categorical variables where OHE is too large but you want less collision than hashing.

**Tip:** Binary encoding reduces dimensionality compared to OHE while keeping more information than label encoding.

In [ ]:
binary_enc = ce.BinaryEncoder(cols=['City'])
df_binary = binary_enc.fit_transform(df.copy())
df_binary.head()

,City_0,City_1,City_2,Education,Gender,Salary,City_LE,Gender_LE,Education_Ordinal,City_Freq,City_Target
0,0,0,1,UG,M,30000,0,1,0.0,2,37865.191481
1,0,1,0,PG,F,45000,2,0,1.0,1,39780.650846
2,0,0,1,UG,M,32000,0,1,0.0,2,37865.191481
3,0,1,1,PhD,F,60000,3,0,2.0,1,41732.277962
4,1,0,0,UG,M,28000,1,1,0.0,1,37568.806782
